## Extended Data Table 4


Enviorment: env 1

In [28]:
import pandas as pd
import pickle
import os
import numpy as np
from sklearn.metrics import confusion_matrix

import sys
sys.path.append("/gpfs/data/shenhavlab/users/ca3261/Visionary-AI-HDP/code/utils/")
from utils import get_test_metrics

#### fmf

In [29]:
fmf_probs = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/fmf_pe_results_104.csv")
fmf_probs["id"] = [id.lower() for id in fmf_probs["patient_id"]]
fmf_probs["first_preg"] = [int(val == "Nulliparous") for val in fmf_probs["input_obhist"]]
fmf_probs["prev_pec"] = [int(val == "Parous, prev PE") for val in fmf_probs["input_obhist"]]

In [30]:
fmf_probs["chtn"] = fmf_probs["input_hyp"].astype(int)

#### maternal information

In [31]:
nyu_clinical = pd.read_excel("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/Visionary AI Participant Tracker (1).xlsx")
nyu_clinical["id"] = [id.lower() for id in nyu_clinical["Study ID"].fillna("nan")]
nyu_clinical["aspirin"] = [int("yes" in str(val).lower()) for val in nyu_clinical["Recommended Aspirin?"]]

#### retinal information

In [32]:
nyu_path = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/nyu/nyu_rg_delivered_testset/"
all_probs = []
all_shap_list = []
for i in range(1, 6):
    path = nyu_path + f"nyu_79_no_repeats_med_pec_pw{i}/meta_all_0.5_oof_bls_pw{i}/XGBoost/"
    name = os.listdir(path)[0]
    with open(path + name, "rb") as f:
        data = pickle.load(f)
        for id in data[5].keys():
            all_shap_list.append({"pw": i,"id": id, "shap": data[5][id]})
    all_probs.append(data[1])
all_probs = pd.DataFrame(all_probs)

In [33]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/nyu_paper_cases.pkl", "rb") as f:
    pec = pickle.load(f)

In [34]:
retinal_probs = pd.DataFrame(all_probs.mean(), columns=["retinal"])
retinal_probs["id"] = retinal_probs.index
retinal_probs = retinal_probs.reset_index(drop=True)
retinal_probs = retinal_probs.merge(fmf_probs[["id", "pe_before_delivery_percent", "cohort", "first_preg", "prev_pec", "input_age", "chtn"]], how="outer")
retinal_probs["label"] = (retinal_probs["cohort"] == "case").astype(int)
retinal_probs = retinal_probs.dropna()

### Metrics

In [35]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/nyu_evaluation.pkl", "rb") as f:
    eval_data = pickle.load(file=f)

In [36]:
ghtn_cases = eval_data["ghtn"]
chtn_cases = eval_data["chtn"]
final_exclusions = eval_data["ex"]

In [37]:
ids = list(set(retinal_probs["id"].values) - set(final_exclusions))

In [38]:
len(ids)

66

In [39]:
fmf_df = get_test_metrics(retinal_probs[retinal_probs["id"].isin(ids)].dropna()["label"].values, (retinal_probs[retinal_probs["id"].isin(ids)].dropna()["pe_before_delivery_percent"]/100).values)


Best threshold by F1: 0.04 (F1=0.44)


In [40]:
fmf_df[fmf_df["fpr"] <= 0.00].sort_values(["tpr", "ppv"], ascending=False).round(2).head(1)

,threshold,fpr,tpr,ppv,npv,adjusted ppv,adjusted npv,f1,tp,fp,tn,fn,balanced acc
14,0.14,0.0,0.23,1.0,0.84,1.0,0.94,0.38,3,0,53,10,0.62


In [41]:
fmf_df[fmf_df["fpr"] <= 0.05].sort_values(["tpr", "ppv"], ascending=False).round(2).head(1)

,threshold,fpr,tpr,ppv,npv,adjusted ppv,adjusted npv,f1,tp,fp,tn,fn,balanced acc
14,0.14,0.0,0.23,1.0,0.84,1.0,0.94,0.38,3,0,53,10,0.62


In [42]:
fmf_df[fmf_df["fpr"] <= 0.1].sort_values(["tpr", "ppv"], ascending=False).round(2).head(1)

,threshold,fpr,tpr,ppv,npv,adjusted ppv,adjusted npv,f1,tp,fp,tn,fn,balanced acc
10,0.1,0.08,0.31,0.5,0.84,0.26,0.94,0.38,4,4,49,9,0.62


In [43]:
fmf_df[fmf_df["fpr"] <= 0.15].sort_values(["tpr", "ppv"], ascending=False).round(2).head(1)

,threshold,fpr,tpr,ppv,npv,adjusted ppv,adjusted npv,f1,tp,fp,tn,fn,balanced acc
8,0.08,0.13,0.38,0.42,0.85,0.2,0.94,0.4,5,7,46,8,0.63


In [44]:
retinal_probs["combo"] = (retinal_probs["prev_pec"] * 0.35 + 0.15 *((retinal_probs["first_preg"]) * (retinal_probs["input_age"] > 37).astype(int)))
ret_df = get_test_metrics(retinal_probs[retinal_probs["id"].isin(ids)].dropna()["label"].values, (retinal_probs[retinal_probs["id"].isin(ids)].dropna()["retinal"] + (retinal_probs[retinal_probs["id"].isin(ids)].dropna()["combo"])).values)


Best threshold by F1: 0.50 (F1=0.62)


In [45]:
ret_df[ret_df["fpr"] == 0.0].sort_values(["tpr", "ppv"], ascending=False).round(2).head(1)

,threshold,fpr,tpr,ppv,npv,adjusted ppv,adjusted npv,f1,tp,fp,tn,fn,balanced acc
72,0.72,0.0,0.31,1.0,0.85,1.0,0.94,0.47,4,0,53,9,0.65


In [46]:
ret_df[ret_df["fpr"] <= 0.05].sort_values(["tpr", "ppv"], ascending=False).round(2).head(1)

,threshold,fpr,tpr,ppv,npv,adjusted ppv,adjusted npv,f1,tp,fp,tn,fn,balanced acc
65,0.65,0.02,0.46,0.86,0.88,0.68,0.95,0.6,6,1,52,7,0.72


In [47]:
ret_df[ret_df["fpr"] <= 0.1].sort_values(["tpr", "ppv"], ascending=False).round(2).head(1)

,threshold,fpr,tpr,ppv,npv,adjusted ppv,adjusted npv,f1,tp,fp,tn,fn,balanced acc
53,0.53,0.08,0.54,0.64,0.89,0.38,0.96,0.58,7,4,49,6,0.73


In [48]:
ret_df[ret_df["fpr"] <= 0.15].sort_values(["tpr", "ppv"], ascending=False).round(2).head(1)

,threshold,fpr,tpr,ppv,npv,adjusted ppv,adjusted npv,f1,tp,fp,tn,fn,balanced acc
50,0.5,0.13,0.69,0.56,0.92,0.31,0.97,0.62,9,7,46,4,0.78


In [49]:
def confusion_matrix_metrics(tn, fp, fn, tp):
    ppv      = tp / (tp + fp)   if (tp + fp) > 0      else float('nan')
    npv      = tn / (tn + fn)   if (tn + fn) > 0      else float('nan')
    tpr      = tp / (tp + fn)   if (tp + fn) > 0      else float('nan')
    fpr      = fp / (fp + tn)   if (fp + tn) > 0      else float('nan')
    tnr      = tn / (tn + fp)   if (tn + fp) > 0      else float('nan')
    fnr      = fn / (fn + tp)   if (fn + tp) > 0      else float('nan')
    f1       = 2 * tp / (2 * tp + fp + fn) if (2*tp + fp + fn) > 0 else float('nan')
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    df = pd.DataFrame([{
        "FPR":       fpr,
        "TPR":       tpr,
        "PPV":       ppv,
        "NPV":       npv,
        'adjusted ppv': tpr * 0.08 / (tpr * 0.08 + (fpr) * (1 - 0.08)),
        'adjusted npv': tnr * (1 - 0.08) / (tnr * (1 - 0.08) + fnr * 0.08),
        "F1":        f1,
        "TP":        tp,
        "FP":        fp,
        "TN":        tn,
        "FN":        fn,
    }])

    return df

In [50]:
# aspirin
tn, fp, fn, tp = confusion_matrix([int(id in pec) for id in nyu_clinical["id"] if id in list(retinal_probs[retinal_probs["id"].isin(ids)].dropna()["id"])], nyu_clinical[nyu_clinical["id"].isin(list(retinal_probs[retinal_probs["id"].isin(ids)].dropna()["id"]))]["aspirin"]).ravel().tolist()
confusion_matrix_metrics(tn, fp, fn, tp).round(2)

,FPR,TPR,PPV,NPV,adjusted ppv,adjusted npv,F1,TP,FP,TN,FN
0,0.47,0.62,0.24,0.85,0.1,0.94,0.35,8,25,28,5
